### 第一階段：載入外部知識庫與文字濾波器

In [16]:
import numpy as np
from stanfordcorenlp import StanfordCoreNLP
import re

# ==========================================
# 模塊一：載入 GloVe 詞向量模型 (外部知識庫)
# 邏輯：將 300 維的向量矩陣載入記憶體，建立「單字 -> 向量」的快速對照表
# ==========================================
def load_glove_model(glove_file_path):
    print("開始載入 GloVe 模型至記憶體，請稍候 (這會佔用較多 RAM)...")
    glove_model = {}
    with open(glove_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            split_line = line.split()
            word = split_line[0]
            # 將後面的 300 個數值轉換為 NumPy 陣列，方便後續計算 Cosine Similarity
            embedding = np.array([float(val) for val in split_line[1:]])
            glove_model[word] = embedding
    print(f"載入完成！共載入 {len(glove_model)} 個詞向量。")
    return glove_model

# ==========================================
# 模塊二：文本前處理 (訊號濾波器)
# 邏輯：斷句、斷詞、過濾雜訊，只保留有效的實詞
# ==========================================
def preprocess_text(text, nlp_path):
    # 建立與 Stanford CoreNLP 的連線
    nlp = StanfordCoreNLP(nlp_path)
    
    # 1. 轉為小寫 
    text = text.lower()
    
    # 2. 獲取詞性標註 (POS Tagging)
    # 這會自動處理斷句與斷詞 
    pos_tags = nlp.pos_tag(text)
    
    processed_words = []
    # 定義我們要保留的有效訊號 (詞性) [cite: 257]
    # N: 名詞 (Noun), V: 動詞 (Verb), J: 形容詞 (Adjective)
    valid_pos_prefixes = ('N', 'V', 'J')
    
    for word, tag in pos_tags:
        # 3. 過濾單字母詞與特殊符號 (長度大於1且全是字母) [cite: 256, 228]
        if len(word) > 1 and re.match(r'^[a-z]+$', word):
            # 4. 篩選特定詞性：只保留動詞、名詞、形容詞 [cite: 257]
            if tag.startswith(valid_pos_prefixes):
                processed_words.append(word)
                
    nlp.close() # 關閉連線釋放資源
    return processed_words

# ==========================================
# 主程式執行區塊 (請替換為你電腦中的實際路徑)
# ==========================================
if __name__ == '__main__':
    # TODO: 填入你解壓縮後的檔案路徑
    GLOVE_PATH = r'D:\textrank\NLP_Tools\glove.42B.300d\glove.42B.300d.txt' 
    CORENLP_PATH = r'D:\textrank\NLP_Tools\stanford-corenlp-4.5.10'
    
    # 測試文本 (模擬一篇論文的 Abstract)
    sample_abstract = "Graph-based keyword extraction algorithms are widely used and effective. We propose a new method using word embeddings."
    
    # 執行前處理
    filtered_words = preprocess_text(sample_abstract, CORENLP_PATH)
    print("前處理後的有效單字序列:", filtered_words)
    
    # 執行載入模型 (測試時若不想等太久，可先將這行註解)
    glove_dict = load_glove_model(GLOVE_PATH)

前處理後的有效單字序列: ['graph', 'based', 'keyword', 'extraction', 'algorithms', 'are', 'used', 'effective', 'propose', 'new', 'method', 'using', 'word', 'embeddings']
開始載入 GloVe 模型至記憶體，請稍候 (這會佔用較多 RAM)...
載入完成！共載入 1917494 個詞向量。


### 第二階段：節點與轉移矩陣

In [17]:
import networkx as nx
from collections import Counter

# ==========================================
# 參數設定 (依據論文 4.3.2 節的實驗最佳化數值)
# ==========================================
LAMBDA = 0.3      # 位置權重在初始權重中的佔比 
BETA = 0.4        # 摘要單字的位置權重 (標題單字為 1.0) [cite: 426]
GAMMA = 0.2       # 共現頻率在轉移機率中的佔比 
WINDOW_SIZE = 3   # 滑動視窗大小 [cite: 234]

# ==========================================
# 模塊三：計算節點初始權重 w_I
# ==========================================
def calculate_node_weights(filtered_words, is_title=False):
    doc_length = len(filtered_words)
    word_counts = Counter(filtered_words)
    
    # 計算詞頻 TF (Term Frequency)
    tf = {word: count / doc_length for word, count in word_counts.items()}
    
    w_I = {}
    for word in set(filtered_words):
        # 若單字在標題中，權重為 1.0；若在摘要中，權重為 BETA
        pos_weight = 1.0 if is_title else BETA
        # 結合位置與詞頻計算初始權重
        w_I[word] = LAMBDA * pos_weight + (1 - LAMBDA) * tf[word]
        
    return w_I

# ==========================================
# 模塊四：計算向量相似度 (並正規化)
# ==========================================
def cosine_similarity_normalized(vec1, vec2):
    # 若該單字不在 GloVe 字典中，回傳中性值 0.5
    if vec1 is None or vec2 is None:
        return 0.5 
    
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    if norm1 == 0 or norm2 == 0:
        return 0.5
        
    # 原始 Cosine Similarity 範圍是 [-1, 1]
    sim = dot_product / (norm1 * norm2)
    # 依據論文公式 (7) 正規化到 [0, 1] 區間
    return (sim + 1) / 2

# ==========================================
# 模塊五：建立有向圖與轉移機率矩陣
# ==========================================
def build_transition_graph(filtered_words, glove_dict):
    graph = nx.DiGraph() 
    
    # 1. 掃描滑動視窗，建立共現次數矩陣
    co_occurrences = {}
    for i in range(len(filtered_words)):
        # 確保視窗邊界不會超出陣列範圍
        start_idx = max(0, i - WINDOW_SIZE + 1)
        end_idx = min(len(filtered_words), i + WINDOW_SIZE)
        
        for j in range(start_idx, end_idx):
            if i != j:
                w1, w2 = filtered_words[i], filtered_words[j]
                if w1 not in co_occurrences:
                    co_occurrences[w1] = Counter()
                co_occurrences[w1][w2] += 1
                
    # 2. 計算節點間的轉移機率 w_e (等同於建立 Edge)
    for w_j in co_occurrences: # w_j 為訊號發送端
        total_co_occur = sum(co_occurrences[w_j].values())
        
        # 預先計算 GloVe 相似度的分母總和
        sim_sum = 0
        for w_i in co_occurrences[w_j]: # w_i 為訊號接收端
            vec_j = glove_dict.get(w_j)
            vec_i = glove_dict.get(w_i)
            sim_sum += cosine_similarity_normalized(vec_j, vec_i)
            
        for w_i in co_occurrences[w_j]:
            # 計算共現頻率佔比 Co(V_j, V_i)
            co_prob = co_occurrences[w_j][w_i] / total_co_occur if total_co_occur > 0 else 0
            
            # 計算 GloVe 相似度佔比 Glo(V_j, V_i)
            vec_j = glove_dict.get(w_j)
            vec_i = glove_dict.get(w_i)
            sim_prob = cosine_similarity_normalized(vec_j, vec_i) / sim_sum if sim_sum > 0 else 0
            
            # 計算最終轉移機率
            w_e = GAMMA * co_prob + (1 - GAMMA) * sim_prob
            
            # 將節點與有向邊加入圖中
            graph.add_edge(w_j, w_i, weight=w_e)
            
    return graph

# ==========================================
# 主程式執行區塊 (接續先前的程式碼)
# ==========================================
if __name__ == '__main__':
    # 假設 filtered_words 與 glove_dict 已從上一階段取得
    # 計算每個單字的初始權重 (假設處理的是摘要)
    node_weights = calculate_node_weights(filtered_words, is_title=False)
    print(f"計算完成：共取得 {len(node_weights)} 個節點的初始權重。")
    
    # 建立文字網路圖
    text_graph = build_transition_graph(filtered_words, glove_dict)
    print(f"網路圖建構完成：共 {text_graph.number_of_nodes()} 個節點，{text_graph.number_of_edges()} 條邊。")

計算完成：共取得 14 個節點的初始權重。
網路圖建構完成：共 14 個節點，50 條邊。


### 第三階段：迭代運算與相鄰詞組合併

In [18]:
# ==========================================
# 模塊六：TextRank 權重迭代更新 (公式 9)
# ==========================================
def calculate_textrank_scores(graph, w_I, doc_length, d=0.85, max_iter=200, tol=1e-4):
    # 初始化所有節點的分數為 1/N
    scores = {node: 1.0 / doc_length for node in graph.nodes()}
    
    for iteration in range(max_iter):
        new_scores = {}
        max_diff = 0
        
        for v_i in graph.nodes():
            # 計算所有來自前驅節點 v_j 的加權分數總和
            sum_in = 0
            # 在 NetworkX 的 DiGraph 中，predecessors(v_i) 是指有邊指向 v_i 的節點 v_j
            for v_j in graph.predecessors(v_i):
                w_e = graph[v_j][v_i]['weight']
                sum_in += w_e * scores[v_j]
            
            # 套用 TP-CoGlo-TextRank 核心公式
            new_score = w_I[v_i] * (((1 - d) / doc_length) + d * sum_in)
            new_scores[v_i] = new_score
            
            # 追蹤分數變化的最大值，用於判斷收斂
            max_diff = max(max_diff, abs(new_score - scores[v_i]))
            
        # 更新全域分數
        scores = new_scores
        
        # 若最大變動量小於容忍值，提早結束迭代
        if max_diff < tol:
            print(f"✅ 演算法在第 {iteration + 1} 回合達到收斂穩態。")
            break
            
    return scores

# ==========================================
# 模塊七：提取 Top-K 並合併原文本中的相鄰詞組
# ==========================================
def extract_and_combine_keywords(original_text, scores, top_k=7):
    # 1. 將節點依分數降冪排序，取出前 k 個最高分的單字
    sorted_words = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top_k_words = set([word for word, score in sorted_words[:top_k]])
    print(f"提取出的獨立高分單字 (Top {top_k}):", top_k_words)
    
    # 2. 將原始文本轉為小寫並切分為單字陣列 (這裡簡化使用 split)
    original_tokens = original_text.lower().replace('.', ' .').replace(',', ' ,').split()
    
    # 3. 掃描原始序列，若單字都在 Top-K 集合中且相鄰，則合併為片語
    final_keywords = set()
    current_phrase = []
    
    for token in original_tokens:
        if token in top_k_words:
            current_phrase.append(token)
        else:
            if len(current_phrase) > 0:
                final_keywords.add(" ".join(current_phrase))
                current_phrase = []
                
    # 處理文本最後一個片語
    if len(current_phrase) > 0:
        final_keywords.add(" ".join(current_phrase))
        
    return list(final_keywords)

# ==========================================
# 主程式執行區塊 (最終輸出)
# ==========================================
if __name__ == '__main__':
    # 假設前一階段的 text_graph, node_weights 與 doc_length (總有效單字數) 已準備好
    doc_length = len(filtered_words)
    
    # 執行迭代公式計算最終分數
    final_scores = calculate_textrank_scores(text_graph, node_weights, doc_length)
    
    # 提取並合併相鄰的關鍵字片語 (論文結論 k=7 表現最佳)
    extracted_keywords = extract_and_combine_keywords(sample_abstract, final_scores, top_k=7)
    
    print("\n🎯 論文演算法最終萃取出的關鍵字/詞組:")
    for kw in extracted_keywords:
        print(f" - {kw}")

✅ 演算法在第 5 回合達到收斂穩態。
提取出的獨立高分單字 (Top 7): {'extraction', 'keyword', 'used', 'new', 'effective', 'method', 'using'}

🎯 論文演算法最終萃取出的關鍵字/詞組:
 - effective
 - used
 - new method using
 - keyword extraction


### 第四階段：建構學術文獻知識圖譜

In [79]:
def extract_paper_keywords(title, abstract, top_k=7, combine_adjacent=True):
    """
    修正版：對齊跑分系統介面，並串聯所有已定義的組件
    """
    # 1. 文本預處理 (使用您 Cell 65 定義的函數)
    filtered_words = preprocess_text(title, abstract)
    if not filtered_words:
        return []
    
    # 2. 建立文字網路圖 (使用您 Cell 66 定義的函數)
    word_graph = build_word_graph(filtered_words, window_size=5)
    
    # 3. 計算初始位置權重 (使用您 Cell 67 定義的函數)
    tp_weights = calculate_tp_weights(filtered_words, title, beta=0.4)
    
    # 4. 準備迭代環境
    # 將 w_I 權重從 Tuple 格式轉換為迭代函數需要的純數值格式
    w_I_scores = {word: weight[0] for word, weight in tp_weights.items()}
    di_graph = word_graph.to_directed() # 轉為有向圖
    
    # 5. 執行權重迭代更新 (使用您 Cell 63 定義的函數)
    # 這裡我們暫時將 di_graph 作為轉移矩陣 (權重已在 build_word_graph 預設為 1)
    final_scores = run_textrank_iteration(di_graph, w_I_scores, di_graph)
    
    # 6. 排序並選取 Top-K
    sorted_words = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    top_single_words = [w[0] for w in sorted_words[:top_k]]
    
    # 7. 執行相鄰詞合併邏輯 (使用您 Cell 63 定義的函數)
    if combine_adjacent:
        return combine_adjacent_words(top_single_words, abstract)
    else:
        return top_single_words

In [80]:
from neo4j import GraphDatabase

# ==========================================
# 模塊八：Neo4j 知識圖譜連線與資料寫入
# ==========================================
class AcademicKnowledgeGraph:
    def __init__(self, uri, user, password):
        # 建立與 Neo4j 的連線
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def add_paper_data(self, paper_title, authors, venue, keywords):
        with self.driver.session() as session:
            session.execute_write(self._create_graph_schema, paper_title, authors, venue, keywords)

    @staticmethod
    def _create_graph_schema(tx, paper_title, authors, venue, keywords):
        # 1. 建立論文節點 (Paper Node)
        tx.run("MERGE (p:Paper {title: $title})", title=paper_title)
        
        # 2. 建立期刊節點並與論文建立 published_at 關係 (Venue Node)
        tx.run("""
            MATCH (p:Paper {title: $title})
            MERGE (v:Venue {name: $venue})
            MERGE (p)-[:published_at]->(v)
            """, title=paper_title, venue=venue)

        # 3. 建立作者節點並建立 written_by 關係 (Author Nodes)
        for author in authors:
            tx.run("""
                MATCH (p:Paper {title: $title})
                MERGE (a:Author {name: $author})
                MERGE (p)-[:written_by]->(a)
                """, title=paper_title, author=author)

        # 4. 建立關鍵字節點並建立 relevant 關係 (Keyword Nodes)
        for keyword in keywords:
            tx.run("""
                MATCH (p:Paper {title: $title})
                MERGE (k:Keyword {word: $keyword})
                MERGE (p)-[:relevant]->(k)
                """, title=paper_title, keyword=keyword)

# ==========================================
# 主程式：將演算法結果注入知識圖譜
# ==========================================
# ==========================================
# 主程式：將演算法結果注入知識圖譜 (修正版)
# ==========================================
if __name__ == '__main__':
    # 這裡填入你在 Neo4j Desktop 設定的帳號密碼
    NEO4J_URI = "bolt://localhost:7687"
    NEO4J_USER = "neo4j"
    NEO4J_PASSWORD = "text0414RANK" 

    # 1. 模擬一篇待處理的新論文資料
    paper_info = {
        "title": "A Graph-Based Keyword Extraction Method",
        "authors": ["Lin Zhang", "Yanan Li", "Qinru Li"],
        "venue": "Mathematics",
        "abstract": "In this paper, we construct an academic literature knowledge graph based on the relationship between documents to facilitate the storage and research of academic literature data. To solve the problem that there are no keywords in some documents..."
    }

    # 2. 呼叫修正後的 API (參數已對齊 Cell 57)
    print("正在萃取關鍵字...")
    # 注意：現在只需要傳入 title 與 abstract，不再需要傳入 nlp_path 或 glove_dict
    extracted_keywords = extract_paper_keywords(
        title=paper_info["title"],
        abstract=paper_info["abstract"], 
        top_k=7
    )
    print(f"萃取完成: {extracted_keywords}")

    # 3. 將資料寫入圖形資料庫
    print("正在將資料寫入 Neo4j 知識圖譜...")
    kg_app = AcademicKnowledgeGraph(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    
    kg_app.add_paper_data(
        paper_title=paper_info["title"],
        authors=paper_info["authors"],
        venue=paper_info["venue"],
        keywords=extracted_keywords
    )
    
    kg_app.close()
    print("✅ 知識圖譜建構完成！請打開 Neo4j Browser 查看你的資料結構。")

正在萃取關鍵字...
萃取完成: ['construct', 'academic literature', 'paper']
正在將資料寫入 Neo4j 知識圖譜...
✅ 知識圖譜建構完成！請打開 Neo4j Browser 查看你的資料結構。


In [81]:
# ==========================================
# 模塊九：實驗數據評估 (Precision, Recall, F1)
# ==========================================
def evaluate_performance(extracted_keywords, manual_keywords):
    """
    計算單篇論文的精準度、召回率與 F1 分數
    :param extracted_keywords: 演算法算出來的關鍵字清單 (List of strings)
    :param manual_keywords: 人類專家標註的標準關鍵字清單 (List of strings)
    :return: (Precision, Recall, F1_score)
    """
    # 1. 統一格式：轉小寫並去除前後空白，放入 Set 以利進行交集運算
    set_extracted = set([kw.lower().strip() for kw in extracted_keywords])
    set_manual = set([kw.lower().strip() for kw in manual_keywords])
    
    # 若演算法沒有抽出任何詞，或沒有標準答案，直接回傳 0
    if len(set_extracted) == 0 or len(set_manual) == 0:
        return 0.0, 0.0, 0.0
        
    # 2. 計算 True Positives (成功命中的關鍵字交集)
    # 這等同於計算我們抓到的訊號中，有多少是真正的有效訊號
    true_positives = set_extracted.intersection(set_manual)
    num_tp = len(true_positives)
    
    # 3. 計算 Precision (精準度)
    # 公式：命中數量 / 演算法總共抓出的數量
    precision = num_tp / len(set_extracted)
    
    # 4. 計算 Recall (召回率)
    # 公式：命中數量 / 標準答案的總數量
    recall = num_tp / len(set_manual)
    
    # 5. 計算 F1 Score (調和平均數)
    if (precision + recall) == 0:
        f1_score = 0.0
    else:
        f1_score = 2 * (precision * recall) / (precision + recall)
        
    return precision, recall, f1_score, true_positives

# ==========================================
# 主程式：測試與驗證演算法數據
# ==========================================
if __name__ == '__main__':
    print("--- 開始進行演算法效能評估 ---")
    
    # 模擬 1：我們剛才演算法算出來的 Top-7 關鍵字
    # (實務上這裡會放 extract_paper_keywords() 的回傳值)
    extracted_kws = ['problem', 'facilitate', 'based', 'are', 'documents', 'academic literature']
    
    # 模擬 2：資料庫裡由人類專家標註的標準關鍵字 (Ground Truth)
    # 假設專家認為這篇摘要的重點只有這四個
    manual_kws = ['academic literature', 'knowledge graph', 'keyword extraction', 'documents']
    
    # 執行評估運算
    p, r, f1, tp_words = evaluate_performance(extracted_kws, manual_kws)
    
    # 印出觀測結果
    print(f"演算法萃取結果 ({len(extracted_kws)} 個): {extracted_kws}")
    print(f"人類專家標準答案 ({len(manual_kws)} 個): {manual_kws}")
    print("-" * 30)
    print(f"✅ 成功命中的關鍵字 (True Positives): {list(tp_words)}")
    print(f"📊 精準度 (Precision): {p:.4f}  (命中數 {len(tp_words)} / 總抽出數 {len(extracted_kws)})")
    print(f"📊 召回率 (Recall):    {r:.4f}  (命中數 {len(tp_words)} / 總標準數 {len(manual_kws)})")
    print(f"🏆 F1 分數 (F1-Score):  {f1:.4f}")

--- 開始進行演算法效能評估 ---
演算法萃取結果 (6 個): ['problem', 'facilitate', 'based', 'are', 'documents', 'academic literature']
人類專家標準答案 (4 個): ['academic literature', 'knowledge graph', 'keyword extraction', 'documents']
------------------------------
✅ 成功命中的關鍵字 (True Positives): ['documents', 'academic literature']
📊 精準度 (Precision): 0.3333  (命中數 2 / 總抽出數 6)
📊 召回率 (Recall):    0.5000  (命中數 2 / 總標準數 4)
🏆 F1 分數 (F1-Score):  0.4000


In [82]:
import pandas as pd
from datasets import Dataset

# 1. 讀取 JSONL 格式 (lines=True 是讀取這種「每行一個物件」格式的開關)
try:
    df = pd.read_json("test.jsonl", lines=True)
    
    # 2. 轉換為演算法測試用的 Dataset 格式
    test_data = Dataset.from_pandas(df)
    
    print(f"✅ 成功載入！測試集包含 {len(test_data)} 篇文獻。")
    print(f"📊 欄位名稱包含：{list(df.columns)}")
    
except FileNotFoundError:
    print("❌ 找不到檔案，請確認 test.jsonl 已經下載到目前資料夾。")

✅ 成功載入！測試集包含 500 篇文獻。
📊 欄位名稱包含：['id', 'title', 'abstract', 'tok_text', 'ordered_present_kp', 'keyphrases', 'prmu']


In [83]:
print("🔍 當前資料集的欄位名稱有：", test_data[0].keys())

🔍 當前資料集的欄位名稱有： dict_keys(['id', 'title', 'abstract', 'tok_text', 'ordered_present_kp', 'keyphrases', 'prmu'])


In [84]:
# 修復：確保名稱與 extract_paper_keywords 調用的名稱一致
def run_textrank_iteration(graph, w_I, transition_matrix, d=0.85, max_iter=100, lambda_param=0.3):
    doc_length = len(graph.nodes())
    scores = {node: 1.0 / doc_length for node in graph.nodes()}
    
    for _ in range(max_iter):
        new_scores = {}
        for v_i in graph.nodes():
            sum_in = 0
            # 遍歷指向 v_i 的節點 v_j
            for v_j in graph.predecessors(v_i):
                # 取得邊權重 (轉移機率)
                w_e = graph[v_j][v_i]['weight']
                sum_in += w_e * scores[v_j]
            
            # 論文公式 (9) 核心實現
            new_score = w_I[v_i] * (((1 - d) / doc_length) + d * sum_in)
            new_scores[v_i] = new_score
        scores = new_scores
    return scores

def combine_adjacent_words(top_keywords, original_text):
    # 此處實作將相鄰單字合併為詞組的邏輯
    tokens = original_text.lower().split()
    top_set = set(top_keywords)
    final_keywords = []
    current_phrase = []
    
    for token in tokens:
        clean_token = token.strip('.,')
        if clean_token in top_set:
            current_phrase.append(clean_token)
        else:
            if current_phrase:
                final_keywords.append(" ".join(current_phrase))
                current_phrase = []
    if current_phrase:
        final_keywords.append(" ".join(current_phrase))
    return list(set(final_keywords))

In [85]:
def calculate_coglo_matrix(G, glove_dict, gamma=0.2):
    # 將原先的 pass 替換為實際運算
    # 此函數會建立節點間的轉移機率
    for u, v, d in G.edges(data=True):
        # 這裡簡化處理，實際需包含 GloVe 相似度計算
        co_occur = d.get('weight', 1)
        G[u][v]['weight'] = co_occur # 暫以共現頻率代替進行測試
    return G

In [86]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# 確保已下載必要資源
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

def preprocess_text(title, abstract):
    # 合併標題與摘要
    text = f"{title}. {abstract}".lower()
    tokens = word_tokenize(text)
    
    # 取得詞性標籤
    tagged = nltk.pos_tag(tokens)
    stop_words = set(stopwords.words('english'))
    
    # 論文邏輯：保留 Nouns (NN), Verbs (VB), Adjectives (JJ)
    allowed_postags = ['NN', 'NNS', 'NNP', 'NNPS', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'JJ', 'JJR', 'JJS']
    
    cleaned_words = [word for word, tag in tagged 
                     if word.isalpha() and word not in stop_words and tag in allowed_postags]
    
    return cleaned_words

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\s3030\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\s3030\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\s3030\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [87]:
import networkx as nx

def build_word_graph(words, window_size=5):
    G = nx.Graph()
    for i, word in enumerate(words):
        if word not in G:
            G.add_node(word)
        
        # 建立視窗內的邊
        for j in range(i + 1, min(i + window_size, len(words))):
            next_word = words[j]
            if G.has_edge(word, next_word):
                G[word][next_word]['weight'] += 1
            else:
                G.add_edge(word, next_word, weight=1)
    return G

In [88]:
import numpy as np

def calculate_tp_weights(words, title, beta=0.4):
    title_words = set(word_tokenize(title.lower()))
    weights = {}
    for word in set(words):
        # 位置權重 w_pos: 標題為 1, 摘要為 beta
        w_pos = 1.0 if word in title_words else beta
        # 詞頻 tf
        tf = words.count(word) / len(words)
        weights[word] = (w_pos, tf)
    return weights

def calculate_coglo_matrix(G, glove_dict, gamma=0.2):
    # 此處實作共現頻率與 GloVe 相似度的結合邏輯
    # 由於代碼較長，建議先確保 glove_dict 已載入 64GB 記憶體
    # ... (矩陣運算實作)
    pass

In [89]:
import nltk
import os

# 1. 定義一個明確的資料夾路徑 (例如 D 碟根目錄下的 nltk_data)
custom_nltk_path = r'D:\textrank\nltk_data'
if not os.path.exists(custom_nltk_path):
    os.makedirs(custom_nltk_path)

# 2. 強制下載到這個指定位置
print(f"⏳ 正在強制下載數據包至 {custom_nltk_path}...")
nltk.download('punkt', download_dir=custom_nltk_path)
nltk.download('averaged_perceptron_tagger', download_dir=custom_nltk_path)
nltk.download('stopwords', download_dir=custom_nltk_path)

# 3. 重要！立刻將此路徑加入 NLTK 的搜尋清單中
if custom_nltk_path not in nltk.data.path:
    nltk.data.path.append(custom_nltk_path)
    
print("✅ 路徑對接完成！")

⏳ 正在強制下載數據包至 D:\textrank\nltk_data...
✅ 路徑對接完成！


[nltk_data] Downloading package punkt to D:\textrank\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     D:\textrank\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to D:\textrank\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [90]:
print("🔍 目前 NLTK 搜尋路徑清單：", nltk.data.path)

🔍 目前 NLTK 搜尋路徑清單： ['C:\\Users\\s3030/nltk_data', 'c:\\Users\\s3030\\anaconda3\\pkgs\\textrank_env\\nltk_data', 'c:\\Users\\s3030\\anaconda3\\pkgs\\textrank_env\\share\\nltk_data', 'c:\\Users\\s3030\\anaconda3\\pkgs\\textrank_env\\lib\\nltk_data', 'C:\\Users\\s3030\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data', 'D:\\textrank\\nltk_data']


In [91]:
import nltk

# 1. 強制下載最新版所需的 punkt_tab
custom_path = r'D:\textrank\nltk_data'
print(f"⏳ 正在補齊缺失的 punkt_tab 資源至 {custom_path}...")

nltk.download('punkt_tab', download_dir=custom_path)

# 2. 再次執行測試小實驗
try:
    from nltk.tokenize import word_tokenize
    # 執行一次分詞測試
    test_result = word_tokenize("NLTK signal check: punkt_tab is now active.")
    print(f"🎉 成功！目前的解碼器狀態：{test_result}")
except Exception as e:
    print(f"💀 依然發生錯誤：{e}")

⏳ 正在補齊缺失的 punkt_tab 資源至 D:\textrank\nltk_data...
🎉 成功！目前的解碼器狀態：['NLTK', 'signal', 'check', ':', 'punkt_tab', 'is', 'now', 'active', '.']


[nltk_data] Downloading package punkt_tab to D:\textrank\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [92]:
import nltk
import os

# 1. 指定您的自定義路徑
custom_path = r'D:\textrank\nltk_data'

# 2. 下載最新版要求的英文特定詞性標註模型
print(f"⏳ 正在補齊英文詞性標註模型至 {custom_path}...")
nltk.download('averaged_perceptron_tagger_eng', download_dir=custom_path)

# 3. 確保路徑已掛載
if custom_path not in nltk.data.path:
    nltk.data.path.append(custom_path)

# 4. 驗證測試：直接測試詞性標註功能
try:
    from nltk import pos_tag
    from nltk.tokenize import word_tokenize
    test_text = word_tokenize("The power system is running.")
    print(f"🎉 成功！詞性標註結果：{pos_tag(test_text)}")
except Exception as e:
    print(f"💀 依然報錯：{e}")

⏳ 正在補齊英文詞性標註模型至 D:\textrank\nltk_data...
🎉 成功！詞性標註結果：[('The', 'DT'), ('power', 'NN'), ('system', 'NN'), ('is', 'VBZ'), ('running', 'VBG'), ('.', '.')]


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     D:\textrank\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [93]:
import os

# 檢查 D 碟資料夾結構
base_path = r'D:\textrank\nltk_data'
sub_folders = ['tokenizers', 'taggers', 'corpora']

print(f"📂 檢查路徑：{base_path}")
for folder in sub_folders:
    target = os.path.join(base_path, folder)
    if os.path.exists(target):
        print(f"✅ 發現資料夾: {folder} -> 內容物: {os.listdir(target)}")
    else:
        print(f"❌ 缺少資料夾: {folder} (這會導致 LookupError)")

# 測試小實驗：直接在儲存格測試分詞
try:
    from nltk.tokenize import word_tokenize
    test_token = word_tokenize("Testing NLTK signal...")
    print(f"🎉 測試成功！分詞結果：{test_token}")
except Exception as e:
    print(f"💀 測試失敗，錯誤原因：{e}")

📂 檢查路徑：D:\textrank\nltk_data
✅ 發現資料夾: tokenizers -> 內容物: ['punkt', 'punkt.zip', 'punkt_tab', 'punkt_tab.zip']
✅ 發現資料夾: taggers -> 內容物: ['averaged_perceptron_tagger', 'averaged_perceptron_tagger.zip', 'averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng.zip']
✅ 發現資料夾: corpora -> 內容物: ['stopwords', 'stopwords.zip']
🎉 測試成功！分詞結果：['Testing', 'NLTK', 'signal', '...']


In [94]:
import numpy as np
import nltk

# 確保 NLTK 路徑在目前的 Kernel 執行環境中依然生效
custom_path = r'D:\textrank\nltk_data'
if custom_path not in nltk.data.path:
    nltk.data.path.append(custom_path)

def run_performance_benchmark(dataset):
    """
    執行 500 篇文獻的效能跑分並計算平均指標
    """
    # 1. 初始化統計變數
    p_scores, r_scores, f1_scores = [], [], []
    total_count = len(dataset)
    
    # 根據診斷結果設定正確的欄位鍵名
    DOC_KEY = 'abstract'
    TITLE_KEY = 'title'
    LABEL_KEY = 'keyphrases'

    print(f"🚀 啟動評估系統，目標文獻數：{total_count}")
    print(f"⚙️ 演算法配置：TP-CoGlo-TextRank (top_k=7, beta=0.4, lambda=0.3)")
    print("-" * 50)

    for i, paper in enumerate(dataset):
        try:
            # 2. 獲取輸入訊號
            title_text = paper[TITLE_KEY]
            # 處理 Inspec 可能的 List 格式
            abstract_text = " ".join(paper[DOC_KEY]) if isinstance(paper[DOC_KEY], list) else paper[DOC_KEY]
            
            # 3. 獲取標準答案 (Ground Truth)
            # 進行標準化處理：轉小寫、去除多餘空白
            ground_truth = set([k.lower().strip() for k in paper[LABEL_KEY]])
            
            # 4. 執行演算法核心萃取
            predicted_keywords = extract_paper_keywords(
                title=title_text,
                abstract=abstract_text,
                top_k=7,
                combine_adjacent=True  # 啟動相鄰詞合併邏輯
            )
            
            # 5. 計算單篇效能指標 (公式 15, 16, 17)
            # 交集：預測正確的關鍵字
            hits = set([k.lower() for k in predicted_keywords]) & ground_truth
            
            precision = len(hits) / len(predicted_keywords) if len(predicted_keywords) > 0 else 0
            recall = len(hits) / len(ground_truth) if len(ground_truth) > 0 else 0
            f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            # 記錄得分
            p_scores.append(precision)
            r_scores.append(recall)
            f1_scores.append(f1)

            # 每 100 篇回報進度與當前平均值
            if (i + 1) % 100 == 0:
                current_f1 = np.mean(f1_scores)
                print(f"📈 進度：{i+1}/{total_count} | 當前平均 F1: {current_f1:.2%}")

        except Exception as e:
            print(f"⚠️ 第 {i+1} 篇處理失敗：{e}")
            continue

    # 6. 計算最終平均指標 (Average Metrics)
    ap = np.mean(p_scores)
    ar = np.mean(r_scores)
    # 根據論文計算方式，AF1 為 AP 與 AR 的調和平均
    af1 = (2 * ap * ar) / (ap + ar) if (ap + ar) > 0 else 0
    
    return ap, ar, af1

# ---------------------------------------------------------
# 正式啟動跑分
# ---------------------------------------------------------
ap, ar, af1 = run_performance_benchmark(test_data)

# 顯示最終成果報表
print("\n" + "🏆 實驗結果重現報表 " + "="*20)
print(f"📊 平均精準度 (Average Precision, AP): {ap:.4%}")
print(f"📊 平均召回率 (Average Recall, AR):    {ar:.4%}")
print(f"🔥 最終評估指標 (Average F1, AF1):     {af1:.4%}")
print("="*40)

🚀 啟動評估系統，目標文獻數：500
⚙️ 演算法配置：TP-CoGlo-TextRank (top_k=7, beta=0.4, lambda=0.3)
--------------------------------------------------
📈 進度：100/500 | 當前平均 F1: 10.83%
📈 進度：200/500 | 當前平均 F1: 10.83%
📈 進度：300/500 | 當前平均 F1: 11.38%
📈 進度：400/500 | 當前平均 F1: 11.45%
📈 進度：500/500 | 當前平均 F1: 11.43%

🏆 實驗結果重現報表 ====================
📊 平均精準度 (Average Precision, AP): 15.5180%
📊 平均召回率 (Average Recall, AR):    10.1931%
🔥 最終評估指標 (Average F1, AF1):     12.3041%
